# YOLO-Bad-Triangle — Adversarial Sweep

Runs the full BUILD → ATTACK → DEFEND sweep on a GPU.

**Before running:** `Runtime → Change runtime type → T4 GPU`

## 1. Check GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU — change runtime type to T4 GPU before continuing.')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Clone Repo & Install Dependencies

In [2]:
import os
REPO_URL = 'https://github.com/Cmaddock99/YOLO-Bad-Triangle.git'
REPO_DIR = '/content/YOLO-Bad-Triangle'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

Working directory: /content/YOLO-Bad-Triangle


In [3]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics>=8.4.23', 'opencv-python-headless', 'pandas', 'tqdm', 'pyyaml'], check=True)
print('Dependencies installed.')

Dependencies installed.


## 3. Download COCO val2017 Subset (500 images)

In [ ]:
import json, urllib.request, zipfile
from pathlib import Path
from tqdm import tqdm

COCO_DIR = Path('coco/val2017_subset500')
IMG_DIR  = COCO_DIR / 'images'
LBL_DIR  = COCO_DIR / 'labels'
IMG_DIR.mkdir(parents=True, exist_ok=True)
LBL_DIR.mkdir(parents=True, exist_ok=True)

ANNO_ZIP  = Path('/tmp/annotations_trainval2017.zip')
ANNO_JSON = Path('/tmp/coco_anno/annotations/instances_val2017.json')
SUBSET_JSON = COCO_DIR / 'instances_val2017_subset500.json'

if not SUBSET_JSON.exists():
    if not ANNO_ZIP.exists():
        print('Downloading COCO annotations (~241 MB)...')
        urllib.request.urlretrieve(
            'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
            ANNO_ZIP)
    if not ANNO_JSON.exists():
        print('Extracting...')
        with zipfile.ZipFile(ANNO_ZIP) as zf:
            zf.extractall('/tmp/coco_anno')
    with open(ANNO_JSON) as f:
        full = json.load(f)
    subset_images = full['images'][:500]
    subset_ids    = {img['id'] for img in subset_images}
    subset_annos  = [a for a in full['annotations'] if a['image_id'] in subset_ids]
    subset = {**full, 'images': subset_images, 'annotations': subset_annos}
    SUBSET_JSON.write_text(json.dumps(subset))
    print(f'Subset saved: {len(subset_images)} images, {len(subset_annos)} annotations')

with open(SUBSET_JSON) as f:
    subset = json.load(f)

missing = [img for img in subset['images'] if not (IMG_DIR / img['file_name']).exists()]
print(f'Downloading {len(missing)} images...')
for img in tqdm(missing, unit='img'):
    urllib.request.urlretrieve(
        'http://images.cocodataset.org/val2017/' + img['file_name'],
        IMG_DIR / img['file_name'])
print(f'Images ready: {len(list(IMG_DIR.iterdir()))}')

 56%|█████▌    | 281/500 [00:51<00:41,  5.27img/s]

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

COCO_DIR = Path('coco/val2017_subset500')
LBL_DIR  = COCO_DIR / 'labels'

with open(COCO_DIR / 'instances_val2017_subset500.json') as f:
    data = json.load(f)

cat_map = {cat['id']: i for i, cat in enumerate(data['categories'])}
annos_by_image = defaultdict(list)
for anno in data['annotations']:
    annos_by_image[anno['image_id']].append(anno)

written = 0
for img in data['images']:
    lbl_path = LBL_DIR / (Path(img['file_name']).stem + '.txt')
    if lbl_path.exists():
        continue
    W, H = img['width'], img['height']
    lines = []
    for anno in annos_by_image[img['id']]:
        if anno.get('iscrowd', 0):
            continue
        x, y, w, h = anno['bbox']
        lines.append(f"{cat_map[anno['category_id']]} {(x+w/2)/W:.6f} {(y+h/2)/H:.6f} {w/W:.6f} {h/H:.6f}")
    lbl_path.write_text('\n'.join(lines))
    written += 1

print(f'Labels written: {written}, total: {len(list(LBL_DIR.iterdir()))}')

## 4. Download Model Weights

In [ ]:
from pathlib import Path
from ultralytics import YOLO
if not Path('yolo26n.pt').exists():
    print('Downloading yolo26n.pt...')
    YOLO('yolo26n.pt')  # ultralytics auto-downloads to cwd
print('yolo26n.pt ready:', Path('yolo26n.pt').exists())

## 5. Sweep Configuration
Edit these variables to control the sweep.

In [ ]:
ATTACKS    = 'fgsm,pgd,blur,deepfool'
DEFENSES   = 'median_preprocess'
MAX_IMAGES = 500
VALIDATION = True
SEED       = 42
print('Attacks: ', ATTACKS)
print('Defenses:', DEFENSES)
print('Images:  ', MAX_IMAGES)
print('Validate:', VALIDATION)

## 6. Run the Sweep

In [ ]:
import subprocess, sys, os
env = os.environ.copy()
env['PYTHONPATH'] = 'src'
cmd = [
    sys.executable, 'scripts/sweep_and_report.py',
    '--attacks',    ATTACKS,
    '--defenses',   DEFENSES,
    '--max-images', str(MAX_IMAGES),
    '--seed',       str(SEED),
    '--python-bin', sys.executable,
    '--preset',     'full',
]
if VALIDATION:
    cmd.append('--validation-enabled')
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, env=env)
print('Exit code:', result.returncode)

## 7. View Report

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
latest = sorted(Path('outputs/framework_reports').iterdir())[-1]
report = latest / 'framework_run_report.md'
print('Report:', report)
display(Markdown(report.read_text()))

## 8. Download Results

In [ ]:
import shutil
from pathlib import Path
from google.colab import files
latest   = sorted(Path('outputs/framework_reports').iterdir())[-1]
zip_path = f'/tmp/{latest.name}'
shutil.make_archive(zip_path, 'zip', latest)
files.download(zip_path + '.zip')